# Assessment 1: Analysing historical data with system performance - Phase 2

**Student ID:** 35721588  
**Unit:** ITO5202  
**Teaching Period:** 5, 2026

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Source:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

---

## Contents

**Part A: Analytical query design and implementation**
1. Business query design and justification
2. DataFrame API implementation
3. Spark SQL implementation
4. Result validation and API comparison

**Part B: System perspective and performance analysis**
1. Partitioning strategy analysis
2. Execution time benchmarking
3. Execution plan interpretation
4. DAG analysis via the Spark Web UI

---

## Environment and configuration

### Execution environment

For this project, we plan to run Spark in **local mode** on a single machine. In this setup, Spark does not create separate executor JVMs. Instead, the driver process handles the computation itself and uses the machine’s available logical CPU cores to run tasks in parallel. Because of this, the main memory setting that matters for our setup is spark.driver.memory, so we do not need to configure executor memory separately.

| Property | Value |
|---|---|
| Machine | MacBook Air (Retina, 13-inch, 2018) |
| Processor | 1.6 GHz dual-core Intel Core i5 |
| Logical cores | 4 |
| Physical memory | 8 GB |
| Operating system | macOS Sonoma 14.7.8 |
| Java | Eclipse Temurin JDK 17 (x64) |
| Python | 3.11.9 |
| PySpark | 3.5.1 |
| Spark master | `local[*]` |

The environment details are generated directly in the notebook rather than written in manually. This means the values referred to later in the Part B benchmarking discussion can be checked against the notebook output.

In [ ]:
# Import packages
import os
import time
from statistics import median
import pandas as pd

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Set up data directory folder path
DATA_DIR = "data"


---

## SparkSession configuration

For this project, there are three key Spark settings which we changed from their default values due to how they affect the behaviour we want to examine later in Part B.

**`spark.driver.memory = 3g`.** Our machine has 8 GB of RAM, which also needs to support the computer's other processes. Allocating too much memory to Spark could slow  down our execution significantly. This matters to us beyond a speed perspective, since Part B.2 compares execution times. More specifically, this would make our results less useful because they could reflect memory pressure rather than Spark's actual processing behaviour.

**`spark.sql.shuffle.partitions = 4`.** This setting determines how many partitions Spark creates after a shuffle. The default is 200, which makes more sense for a much larger cluster than for the local environment we are using here. With four available task slots and a dataset of around 113,000 rows at its largest, using 200 partitions would create many very small tasks and add unnecessary scheduling overhead.

**`spark.sql.adaptive.enabled = false`.** Spark's Adaptive Query Execution (AQE) can change the physical execution plan while a query is running. On one hand, this can improve performance, but on the other hand, it makes the execution harder to compare with the plan shown by `explain(extended=True)`. Because Parts B.3 and B.4 require us to examine the physical plan and its corresponding DAG, AQE is turned off so that the printed plan and the executed plan remain consistent. This also means that the partition counts discussed in Part B.1 reflect the values we set orselves rather than values Spark changes during execution.


In [ ]:
# Spark session build with local mode, 4 task slots, AQE off (as explained above)
spark = SparkSession.builder \
    .appName("ITO5202-A1-Olist-Freight") \
    .master("local[*]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", 4) \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark

In [ ]:
sc = spark.sparkContext

# Print the main Spark environment settings
print("Spark version:", spark.version)
print("Master:", sc.master)
print("Available task slots:", sc.defaultParallelism)
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Broadcast join threshold (bytes):", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Web UI link
print("Web UI:", sc.uiWebUrl)

In [ ]:
# Confirms the session can run a job end to end
spark.range(10).count()


---

## Data Loading

### Defining schemas

Our nine source files are loaded in using manually defined schemas instead of `inferSchema=True`. This is because by us using schema inference, Spark has to inspect the data before it is able to load it, thus adding extra work (this is particularly important for the geolocation file, which contains over one million rows).

Furthermore, if we were to allow Spark to infer the schema automatically, the same column in different data files could be interpreted as being of different data types, which can have downstream impacts on analysis when we try to do joins. Instead, by defining the schema ourselves, we are able to ensure consistency in data types. 

We also note that as per our proposal, we are looking to only use seven of the nine available data files, as do not wish to include the payments and reviews dataset as they fall outside the scope of our analysis. As such, we do not read in these datasets such as to avoid unneccessary extra work.

#### _Note_

Given a heading error in Section 3 of our original proposal, we need to make a correction in our dataset list. The first table in Section 3 of our approved proposal is labelled `olist_geolocation_dataset.csv`, but the columns listed underneath it actually belong to `olist_order_items_dataset.csv`. The geolocation dataset then appears again later in the sme section of our proposal with the correct columns.

We note that the approved proposal has been kept unchanged in `proposal/proposal.md` for consistency. However, the schemas defined below use the actual, correct columns from each source file, which were checked against the downloaded dataset.

#### _Note 2_

The source Kaggle file uses the column names `product_name_lenght` and `product_description_lenght`. However, in our approved proposal, we listed these names with the correct spelling of "length". 

Now, for the purposes of our analysis, the schema needs to match the actual CSV headers exactly, thus creating a mismatch with our earlier proposal. To keep the rest of the notebook easier to read, these two columns are renamed immediately after loading.

In [ ]:
# Define the schemas manually so Spark does not have to infer them

order_items_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_item_id", IntegerType(), False),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])

sellers_schema = StructType([
    StructField("seller_id", StringType(), False),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True),
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", IntegerType(), True),
    StructField("geolocation_lat", DoubleType(), True),
    StructField("geolocation_lng", DoubleType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True),
])

category_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
])

In [ ]:
# Helper function to read a CSV
def load_csv(filename, schema):
    return (spark.read
            .option("header", True)
            .schema(schema)
            .csv(f"{DATA_DIR}/{filename}"))

order_items = load_csv("olist_order_items_dataset.csv", order_items_schema)
orders      = load_csv("olist_orders_dataset.csv", orders_schema)
customers   = load_csv("olist_customers_dataset.csv", customers_schema)
sellers     = load_csv("olist_sellers_dataset.csv", sellers_schema)
geolocation = load_csv("olist_geolocation_dataset.csv", geolocation_schema)
categories  = load_csv("product_category_name_translation.csv", category_schema)

# Correct the misspelled column names present in the source file headers
products = (load_csv("olist_products_dataset.csv", products_schema)
            .withColumnRenamed("product_name_lenght", "product_name_length")
            .withColumnRenamed("product_description_lenght", "product_description_length"))

In [ ]:
# Check row counts against the expected values
order_items_count = order_items.count()
orders_count = orders.count()
customers_count = customers.count()
sellers_count = sellers.count()
geolocation_count = geolocation.count()
products_count = products.count()
categories_count = categories.count()

rows = [
    {
        "dataset": "order_items",
        "expected": 112650,
        "actual": order_items_count,
        "match": order_items_count == 112650,
        "columns": len(order_items.columns)
    },
    {
        "dataset": "orders",
        "expected": 99441,
        "actual": orders_count,
        "match": orders_count == 99441,
        "columns": len(orders.columns)
    },
    {
        "dataset": "customers",
        "expected": 99441,
        "actual": customers_count,
        "match": customers_count == 99441,
        "columns": len(customers.columns)
    },
    {
        "dataset": "sellers",
        "expected": 3095,
        "actual": sellers_count,
        "match": sellers_count == 3095,
        "columns": len(sellers.columns)
    },
    {
        "dataset": "geolocation",
        "expected": 1000163,
        "actual": geolocation_count,
        "match": geolocation_count == 1000163,
        "columns": len(geolocation.columns)
    },
    {
        "dataset": "products",
        "expected": 32951,
        "actual": products_count,
        "match": products_count == 32951,
        "columns": len(products.columns)
    },
    {
        "dataset": "categories",
        "expected": 71,
        "actual": categories_count,
        "match": categories_count == 71,
        "columns": len(categories.columns)
    }
]

pd.DataFrame(rows)


---

## Data quality assessment

Our approved proposal identified a few data quality issues that we will need to check before we can safely begin our analysis. 

As such, it is important for us to now look at these issues in the loaded data, measure how much of the data is affected, and consider the cleaning and aggregation choices we will need to make later when building our main analysis dataset.


In [ ]:
# Check the different order statuses
orders.groupBy("order_status") \
      .count() \
      .orderBy(F.desc("count")) \
      .show()

In [ ]:
# Check missing delivery-related timestamps
orders.select(
    F.sum(F.col("order_approved_at").isNull().cast("int")).alias("missing_approved"),
    F.sum(F.col("order_delivered_carrier_date").isNull().cast("int")).alias("missing_carrier"),
    F.sum(F.col("order_delivered_customer_date").isNull().cast("int")).alias("missing_delivered"),
    F.sum(F.col("order_estimated_delivery_date").isNull().cast("int")).alias("missing_estimated")
).show()

In [ ]:
# Check how many geolocation rows there are for each postcode prefix
geo_stats = geolocation.groupBy("geolocation_zip_code_prefix").count()

total_geo_rows = geolocation.count()
distinct_postcodes = geo_stats.count()

print("Total geolocation rows:", total_geo_rows)
print("Distinct postcode prefixes:", distinct_postcodes)

geo_stats.agg(
    F.avg("count").alias("average rows per postcode"),
    F.max("count").alias("maximum rows per postcode")
).show()

In [ ]:
# Number of different customer postcode prefixes
customer_postcodes = customers.select("customer_zip_code_prefix").distinct().count()

print("Distinct customer postcode prefixes:", customer_postcodes)


# Check how customers are distributed across states
customers.groupBy("customer_state") \
         .count() \
         .orderBy(F.desc("count")) \
         .show(10)

In [ ]:
# Check for missing product information used later in the analysis

products.select(
    F.sum(F.col("product_category_name").isNull().cast("int")).alias("missing_category"),
    F.sum(F.col("product_weight_g").isNull().cast("int")).alias("missing_weight"),
    F.sum(F.col("product_length_cm").isNull().cast("int")).alias("missing_length"),
    F.sum(F.col("product_height_cm").isNull().cast("int")).alias("missing_height"),
    F.sum(F.col("product_width_cm").isNull().cast("int")).alias("missing_width")
).show()


---

## Data quality findings

**Order completeness:** There are 99,441 orders in total, with 96,478 marked as `delivered` (97.0%). However, 2,965 orders have missing `order_delivered_customer_date`, which exceeds the number of non-delivered orders by two. This means that in our data, there are two orders which were marked as delivered even though there was no recorded delivery timestamp. Furthermore, there is no indication from the data as to what could be causing this (e.g. delivery error, system issue, , etc.).
Since we do not have a clear cause, we should treat this as a data integrity issue, and thus want to filter our analysis for both `oder_status = 'delivered'`, as well as a non-null delivery timestamp. 

**Geolocation duplication:** The geolocation dataset has 1,000,163 rows, however it only contains 19,015 unique postcode prefixes. That means that on average, each unique postcode appears ~52.6 times in the dataset, with the most common postcode appearing 1,146 times. Given this, if we were to join this dataset directly, it would result in a large number of duplicate matches.
Thus, we want to reduce our geolocation data to one row per postcode prefix before conducting any joins. As an additional benefit, at this smaller size, it is also small enough to be used as a broadcast lookup rather than requiring both sides of the join to be shuffled.

**Cardinality of partitioning column:** `customer_zip_code_prefix` contains 14,994 unique values, which we can see is very close to the ~15,000 expected values we mentioned in the proposal. Another thing we briefly mentioned in the proposal was the uneven distribution of customers across states. 
From our above data exploration, we can see a much clearer picture of the customer distribution, with São Paulo containing 41,746 customers, which accounts for ~42% of the total, while SP, RJ and MG together account for aother ~66.6%. 
This uneven distribution is important for us later when we consider the hash and range partitioning comparison in Part B.1, because range partitioning may produce less balanced partitions when the values themselves are unevenly distributed.

**Product attribute completeness:** We can see that there are 610 products (1.9%) with no category name, while two products are missing physical dimension values. This means measures that depend on package volume or weight cannot be calculated for those rows. 
As such, instead of filling in estimated values, we want to exclude these rows from the relevant weight-based calculations, with the number of affected records being reported with the results.